In [2]:

import pandas as pd
import re
import time
from playwright.sync_api import sync_playwright


In [ ]:

# =====================================================
# CONFIGURACIÓN
# =====================================================

QUERY = """
(
"migración venezolana"
OR "migrantes venezolanos"
OR "venezolanos en Colombia"
OR "venezolanos en"
OR "refugiados venezolanos"
OR "crisis migratoria venezolana"
)
lang:es
"""

MAX_TWEETS = 1000

OUTPUT_FILE = "x_migracion_venezolana.csv"

In [4]:


# =====================================================
# FUNCIONES
# =====================================================

def limpiar_texto(texto):

    if texto is None:
        return ""

    texto = str(texto)

    texto = texto.replace("\n", " ")
    texto = texto.replace("\r", " ")

    texto = re.sub(r"\s+", " ", texto)

    return texto.strip()


def generar_titulo(texto, longitud=100):

    texto = limpiar_texto(texto)

    if len(texto) <= longitud:
        return texto

    return texto[:longitud] + "..."


# =====================================================
# EXTRACCIÓN
# =====================================================

registros = []

with sync_playwright() as p:

    browser = p.chromium.launch(
        headless=False
    )

    context = browser.new_context()

    page = context.new_page()

    query_url = QUERY.replace(" ", "%20")
    query_url = query_url.replace("\n", "%20")

    url_busqueda = (
        f"https://x.com/search?q={query_url}"
        "&src=typed_query&f=live"
    )

    page.goto(
        url_busqueda,
        wait_until="domcontentloaded"
    )

    print("\n" + "="*60)
    print("SI X SOLICITA LOGIN:")
    print("1. Inicia sesión manualmente.")
    print("2. Espera a que aparezcan los resultados.")
    print("3. Vuelve aquí y presiona ENTER.")
    print("="*60)

    page.goto(url_busqueda)
    time.sleep(2)

    urls_vistas = set()
    scrolls_sin_nuevos = 0

    while len(registros) < MAX_TWEETS:

        tweets = page.locator(
            'article[data-testid="tweet"]'
        )

        cantidad = tweets.count()

        nuevos = 0

        for i in range(cantidad):

            if len(registros) >= MAX_TWEETS:
                break

            try:

                tweet = tweets.nth(i)

                texto = limpiar_texto(
                    tweet.inner_text(timeout=2000)
                )

                if not texto:
                    continue

                links = tweet.locator(
                    'a[href*="/status/"]'
                )

                if links.count() == 0:
                    continue

                url = links.first.get_attribute(
                    "href"
                )

                if not url:
                    continue

                if url.startswith("/"):
                    url = "https://x.com" + url

                if url in urls_vistas:
                    continue

                urls_vistas.add(url)

                nuevos += 1

                # -----------------------------
                # FECHA
                # -----------------------------

                fecha = ""

                try:
                    fecha = (
                        tweet
                        .locator("time")
                        .first
                        .get_attribute("datetime")
                    )
                except:
                    pass

                # -----------------------------
                # AUTOR
                # -----------------------------

                autor = ""

                try:
                    autor_raw = (
                        tweet
                        .locator('[data-testid="User-Name"]')
                        .first
                        .inner_text()
                    )

                    lineas = autor_raw.split("\n")

                    if len(lineas) >= 2:
                        autor = lineas[1]
                    else:
                        autor = lineas[0]

                except:
                    pass

                registros.append({
                    "fecha": fecha,
                    "autor": autor,
                    "pais": "",
                    "medio": "X",
                    "titulo": generar_titulo(texto),
                    "cuerpo_textual": texto,
                    "url": url
                })

            except:
                pass

        print(
            f"Tweets recopilados: {len(registros)}"
        )

        if nuevos == 0:
            scrolls_sin_nuevos += 1
        else:
            scrolls_sin_nuevos = 0

        if scrolls_sin_nuevos >= 5:
            print("No se encontraron más resultados.")
            break

        page.mouse.wheel(0, 15000)

        time.sleep(3)

    browser.close()

# =====================================================
# DATAFRAME
# =====================================================

df = pd.DataFrame(registros)

# =====================================================
# ELIMINAR DUPLICADOS
# =====================================================

df = df.drop_duplicates(
    subset=["url"]
)

# =====================================================
# FECHA
# =====================================================

df["fecha"] = pd.to_datetime(
    df["fecha"],
    errors="coerce",
    utc=True
)

# =====================================================
# LIMPIEZA FINAL
# =====================================================

df["autor"] = df["autor"].fillna("")
df["pais"] = df["pais"].fillna("")
df["medio"] = "X"

df = df.sort_values(
    by="fecha",
    ascending=False
)

# =====================================================
# ORDENAR COLUMNAS
# =====================================================

df = df[
    [
        "fecha",
        "autor",
        "pais",
        "medio",
        "titulo",
        "cuerpo_textual",
        "url"
    ]
]

# =====================================================
# EXPORTAR CSV
# =====================================================

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

# =====================================================
# RESUMEN
# =====================================================

print("\n" + "="*60)
print(f"Registros extraídos: {len(df)}")
print(f"Archivo generado: {OUTPUT_FILE}")
print("="*60)

display(df.head())

Error: It looks like you are using Playwright Sync API inside the asyncio loop.
Please use the Async API instead.